# 05 — Final Supervised Baselines

## COde Six-Label Multimodal Diagnostic Benchmark

This notebook documents the **final corrected supervised baseline experiments** conducted on the COde dental dataset.

The purpose of this stage is to establish a controlled supervised benchmark for the six-label dental diagnosis task before evaluating self-supervised multimodal representation learning and robustness under missing modalities.

The benchmark evaluates the contribution of three information sources:

* **Intraoral photographs**
* **Radiographs**
* **Clinical text**

Seven supervised scenarios are evaluated:

1. Text-only
2. Photograph-only
3. Radiograph-only
4. Photograph + Text
5. Photograph + Radiograph
6. Text + Radiograph
7. Photograph + Radiograph + Text (Full Multimodal)

All seven scenarios are evaluated under the **same final experimental protocol and the same complete-case patient/visit population** to ensure a directly comparable modality ablation.

> **Important:** Results from earlier exploratory baseline experiments used different populations, hyperparameters, or encoder-training settings. Those historical results are retained only as development history and are **not used as the final thesis baseline comparison**.


## 05.1 — Research Scope

The supervised baseline stage answers a preliminary empirical question:

> **How much diagnostic information is available from each modality, and how does supervised multimodal fusion compare with the strongest unimodal baseline?**

This stage is intentionally separated from the later self-supervised experiments.

The objective here is **not yet** to demonstrate robustness to naturally missing radiographs. Instead, the goal is to establish a reliable reference point against which the later SSL and missing-modality experiments can be compared.

The experimental logic is:

**Dataset Audit → Controlled Supervised Baselines → SSL Representation Learning → Missing-Modality Evaluation**

The final research question should therefore be determined from the evidence produced by these stages rather than assumed in advance.


## 05.2 — Diagnostic Task Definition

The task is a **six-label multi-label classification problem**.

Each visit is associated with six binary diagnostic labels:

| Label                        | Target               |
| ---------------------------- | -------------------- |
| `label_caries`               | Dental caries        |
| `label_gingivitis`           | Gingivitis           |
| `label_malocclusion`         | Malocclusion         |
| `label_pulpitis`             | Pulpitis             |
| `label_tooth_loss`           | Tooth loss           |
| `label_tooth_structure_loss` | Tooth structure loss |

Because multiple conditions may occur in the same visit, the labels are modeled independently using a multi-label classification objective.

The primary training loss is:

`BCEWithLogitsLoss`


## 05.3 — Dataset and Fixed Patient-Level Split

The authoritative six-label dataset is:

`results/six_label_patient_level_dataset/labeled_dataset.csv`

The patient-level split was established earlier and is treated as **fixed throughout the supervised and subsequent experiments**.

| Split      | Patients | Visits |
| ---------- | -------: | -----: |
| Train      |    3,360 |  6,129 |
| Validation |      720 |  1,330 |
| Test       |      720 |  1,316 |

The patient-level split is never regenerated during baseline experiments.

This prevents information from the same patient appearing across different splits and ensures that all seven modality scenarios are evaluated using the same underlying partition.


In [12]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

DATASET_PATH = (
    PROJECT_ROOT
    / "results"
    / "six_label_patient_level_dataset"
    / "labeled_dataset.csv"
)

df = pd.read_csv(DATASET_PATH)

print("Dataset path:", DATASET_PATH)
print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Dataset path: /home/ubuntu/Projects/thesis-code/results/six_label_patient_level_dataset/labeled_dataset.csv
Dataset shape: (8775, 46)
Columns: 46


## 05.4 — Common Complete-Case Population

For the final supervised comparison, all seven scenarios are trained and evaluated on the same **complete-case population**.

A visit is eligible for this benchmark when it contains:

* at least one photograph,
* at least one radiograph,
* and at least one usable clinical-text field from the approved text fields.

This produces the following authoritative population:

| Split      | Complete-case visits |
| ---------- | -------------------: |
| Train      |                2,935 |
| Validation |                  627 |
| Test       |                  633 |
| **Total**  |            **4,195** |

### Why use the same population?

A modality-ablation comparison should differ only in **which modalities are provided to the model**, not in which patients or visits are included.

For example, the text-only model does not use the images as input, but it is still evaluated on the same 633 test visits used by the full multimodal model.

Likewise, the radiograph-only model is evaluated on those same complete-case visits.

This makes the seven scenarios directly comparable at the population level.

> **Important:** Earlier modality-specific experiments used larger modality-dependent populations. Those results are not used for the final comparison because the evaluation populations were different.


## 05.5 — Final Baseline Protocol

The next section defines the common training and evaluation protocol used for **all seven final supervised scenarios**.

No scenario-specific hyperparameter tuning is performed for the final benchmark.

The purpose is to ensure that observed differences are primarily attributable to the available modality combination rather than differences in training configuration.


## 05.6 — Final Common Training Protocol

All seven supervised scenarios use the same final training configuration.

| Setting                      | Final value                            |
| ---------------------------- | -------------------------------------- |
| Loss                         | `BCEWithLogitsLoss`                    |
| Batch size                   | 8                                      |
| Maximum epochs               | 50                                     |
| Learning rate                | `2e-5`                                 |
| Weight decay                 | `1e-4`                                 |
| Early stopping patience      | 10                                     |
| Optimizer                    | AdamW                                  |
| Random seed                  | 42                                     |
| Device                       | CUDA                                   |
| Image size                   | 224 × 224                              |
| Image encoder                | ResNet-50                              |
| Image encoder initialization | Pretrained                             |
| Image encoder training       | Frozen                                 |
| Text encoder                 | DistilBERT (`distilbert-base-uncased`) |
| Maximum text length          | 256 tokens                             |
| Hidden dimension             | 256                                    |
| DataLoader workers           | 0                                      |
| Pin memory                   | True                                   |

The same configuration is used for every modality scenario.

The image encoder is initialized from pretrained weights and remains frozen during supervised baseline training.

The text encoder and scenario-specific fusion/classification layers are trainable.

No test-set information is used for model selection or hyperparameter selection.


## 05.7 — Checkpoint Selection

The best model checkpoint is selected **exclusively using validation Macro F1 at the default threshold of 0.5**.

For each training epoch:

1. The model is trained on the training split.
2. Validation predictions are generated.
3. Validation Macro F1 is computed using threshold `0.5`.
4. The checkpoint is saved only when validation Macro F1 improves.
5. Early stopping is applied using a patience of 10 epochs.

Formally:

$$
\text{Best Checkpoint}
=
\arg\max_{e}
\text{MacroF1}_{validation,e}(t=0.5)
$$

This criterion is identical for all seven scenarios.

### Why Macro F1?

The task is multi-label and the six diagnostic labels are not equally balanced. Macro F1 gives each diagnostic label equal weight and therefore provides a useful primary checkpoint-selection criterion for the multi-label task.

The test set is **not consulted during checkpoint selection**.


## 05.8 — Threshold Policy

Two evaluation settings are reported:

### Primary evaluation — Default threshold

The primary thesis comparison uses:

`threshold = 0.5`

for every label.

This provides a common, fixed decision rule across all seven scenarios and avoids using test data to determine the classification threshold.

### Secondary evaluation — Validation-optimized thresholds

As a secondary analysis, one threshold is independently optimized for each diagnostic label using the **validation split only**.

The threshold search grid is:

`0.05, 0.10, 0.15, ..., 0.90, 0.95`

For each label, the threshold producing the highest validation F1 is selected.

These thresholds are then applied **once** to the held-out test predictions.

Therefore:

* test labels are never used to select thresholds;
* test predictions are not used to tune thresholds;
* AUROC is unaffected by threshold selection;
* the default-threshold results remain the primary benchmark.

The optimized-threshold results are reported only as a secondary diagnostic analysis.


## 05.9 — Evaluation Metrics

The following metrics are reported on the held-out test set:

### Primary metrics

* **Macro F1**
* **AUROC**

### Additional metrics

* Micro F1
* Accuracy

Macro F1 is the primary threshold-dependent metric because it weights all six diagnostic labels equally.

AUROC is reported because it evaluates ranking quality independently of the selected classification threshold.

For the final thesis comparison, the main table therefore uses:

> **Test Macro F1 @ threshold 0.5 + Test Macro AUROC**

Optimized-threshold Macro F1 is reported separately and is not used to replace the primary result.


## 05.10 — Frozen Image Encoder and Batch Normalization

The ResNet-50 image encoder is pretrained and frozen during supervised baseline training.

Freezing is implemented at two levels:

1. ResNet parameters have `requires_grad=False`.
2. When the overall model enters training mode, the frozen ResNet remains in evaluation mode.

The second point is important because ResNet-50 contains Batch Normalization layers.

If a frozen encoder is left in training mode, its BatchNorm running statistics can still change even though its parameters are not updated by gradient descent.

The final implementation therefore explicitly keeps the frozen image encoder in evaluation mode during training.

This ensures that the visual representation remains fixed across training epochs and that the supervised baseline comparison is not affected by unintended BatchNorm-statistics updates.

> **Correction:** Results generated before this BatchNorm-freezing correction are considered development results and are not used as final thesis baseline results.


In [13]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /home/ubuntu/Projects/thesis-code
src exists: True


In [14]:
from src.baseline.final import config

print("Final supervised baseline configuration")
print("-" * 50)

print("Dataset:", config.DATASET_PATH)
print("Image root:", config.IMAGE_ROOT)

print("\nScenarios:")
for name, modalities in config.SCENARIOS.items():
    print(f"  {name:20s}: {modalities}")

print("\nTraining:")
print("  Batch size:", config.BATCH_SIZE)
print("  Max epochs:", config.NUM_EPOCHS)
print("  Learning rate:", config.LEARNING_RATE)
print("  Weight decay:", config.WEIGHT_DECAY)
print("  Early stopping patience:", config.EARLY_STOPPING_PATIENCE)

print("\nModel:")
print("  Image pretrained:", config.PRETRAINED_IMAGE_ENCODER)
print("  Image frozen:", config.FREEZE_IMAGE_ENCODER)
print("  Text model:", config.TEXT_MODEL_NAME)
print("  Text max length:", config.TEXT_MAX_LENGTH)
print("  Image size:", config.IMAGE_SIZE)

print("\nEvaluation:")
print("  Default threshold:", config.DEFAULT_THRESHOLD)
print(
    "  Threshold grid:",
    config.THRESHOLD_MIN,
    "to",
    config.THRESHOLD_MAX,
    "step",
    config.THRESHOLD_STEP,
)

print("\nSeed:", config.SEED)
print("Device:", config.DEVICE)

Final supervised baseline configuration
--------------------------------------------------
Dataset: /home/ubuntu/Projects/thesis-code/results/six_label_patient_level_dataset/labeled_dataset.csv
Image root: /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/Images

Scenarios:
  text_only           : ('text',)
  image_only          : ('photograph',)
  xray_only           : ('radiograph',)
  image_text          : ('photograph', 'text')
  image_xray          : ('photograph', 'radiograph')
  text_xray           : ('text', 'radiograph')
  full_multimodal     : ('photograph', 'radiograph', 'text')

Training:
  Batch size: 8
  Max epochs: 50
  Learning rate: 2e-05
  Weight decay: 0.0001
  Early stopping patience: 10

Model:
  Image pretrained: True
  Image frozen: True
  Text model: distilbert-base-uncased
  Text max length: 256
  Image size: 224

Evaluation:
  Default threshold: 0.5
  Threshold grid: 0.05 to 0.95 step 0.05

Seed: 42
Device: cuda


## 05.11 — Final Scenario Definitions

The final benchmark consists of the following seven modality configurations:

| Scenario                | Photograph | Radiograph | Clinical Text |
| ----------------------- | :--------: | :--------: | :-----------: |
| Text-only               |      —     |      —     |       ✓       |
| Photograph-only         |      ✓     |      —     |       —       |
| Radiograph-only         |      —     |      ✓     |       —       |
| Photograph + Text       |      ✓     |      —     |       ✓       |
| Photograph + Radiograph |      ✓     |      ✓     |       —       |
| Text + Radiograph       |      —     |      ✓     |       ✓       |
| Full Multimodal         |      ✓     |      ✓     |       ✓       |

The term **image** refers specifically to the photograph modality in this notebook.

The term **xray** refers specifically to the radiograph modality.

Clinical text is represented using the approved clinical fields described in the next section.


## 05.12 — Clinical Text Input Policy

The clinical-text modality is constructed exclusively from the following four fields:

* `chief_complaint`
* `present_illness`
* `past_medical_record`
* `examination`

These fields are concatenated into a single clinical-text input and encoded using DistilBERT.

The following fields are **excluded** from the model input:

* `anomalies_en`
* `diagnosis`
* `treatment_plan`
* `management`

### Rationale

The excluded fields contain information that is too closely related to the diagnostic targets or may directly encode the clinical diagnosis.

In particular, `anomalies_en` is excluded because it can contain explicit or near-explicit descriptions of abnormalities used to construct the diagnostic labels.

Including such fields would make the text modality artificially informative and could introduce label leakage.

The final text-only and multimodal baselines therefore use only the four approved clinical-history/examination fields.


## 05.13 — Interpreting the Text-Only Baseline

The text-only baseline should **not** be interpreted as a pure symptom-based diagnostic model.

The permitted clinical fields contain history and examination information that may include clinically meaningful findings related to the target conditions.

Furthermore, the six diagnostic labels were derived from clinical information available in the dataset.

Therefore, the text-only result should be interpreted as an **upper-bound clinical-information baseline** rather than as evidence that diagnosis can be performed from symptoms alone.

Its purpose in this benchmark is to quantify how much predictive information is already available from the structured clinical narrative and to provide a strong reference point for evaluating the additional contribution of visual and radiographic information.


## 05.14 — Final Model Architecture

A single modality-aware model implementation is used across all seven supervised scenarios.

The architecture consists of modality-specific encoders followed by scenario-specific classification/fusion layers.

### Photograph branch

Photographs are encoded using a pretrained ResNet-50 encoder.

The resulting representation has dimensionality:

`2048`

The ResNet-50 encoder is frozen during supervised training.

### Radiograph branch

Radiographs use the same ResNet-50 encoder architecture and produce:

`2048`

features.

The radiograph encoder is also frozen.

### Clinical-text branch

Clinical text is encoded using:

`distilbert-base-uncased`

The pooled text representation has dimensionality:

`768`

The text encoder is trainable.

### Classification head

The modality representations are combined according to the selected scenario.

The final hidden dimension of the classification/fusion layers is:

`256`

The classifier produces six logits corresponding to the six diagnostic labels.


## 05.15 — Scenario-Specific Input Dimensions

Before the final classification head, the raw modality representations have the following dimensionalities:

| Scenario                |           Representation |
| ----------------------- | -----------------------: |
| Text-only               |                      768 |
| Photograph-only         |                     2048 |
| Radiograph-only         |                     2048 |
| Photograph + Text       |        2048 + 768 = 2816 |
| Photograph + Radiograph |       2048 + 2048 = 4096 |
| Text + Radiograph       |        768 + 2048 = 2816 |
| Full Multimodal         | 2048 + 2048 + 768 = 4864 |

These dimensions refer to the encoder outputs before the scenario-specific classification/fusion layers.

> **Important correction:** An earlier notebook description stated that each image branch was projected to 512 dimensions before full multimodal fusion, resulting in a `1792`-dimensional input. That description belongs to an earlier architecture and is **not the final supervised baseline architecture**.


## 05.16 — Final Parameter Counts

The number of trainable and frozen parameters depends on the modality combination.

The final recorded parameter counts are:

| Scenario          | Total parameters |  Trainable |     Frozen |
| ----------------- | ---------------: | ---------: | ---------: |
| Text-only         |       66,561,286 | 66,561,286 |          0 |
| Radiograph-only   |       24,034,118 |    526,086 | 23,508,032 |
| Text + Radiograph |       90,593,606 | 67,085,574 | 23,508,032 |
| Full Multimodal   |      114,625,926 | 67,609,862 | 47,016,064 |

For the photograph-only and photograph-containing scenarios, the same ResNet-50 encoder structure is used for the photograph branch.

The important architectural principle is that pretrained image encoders are frozen, whereas the text encoder and task-specific classification/fusion layers remain trainable.

Parameter counts are reported for transparency and should not be interpreted as a measure of model quality.


In [15]:
from src.baseline.final.config import (
    LABEL_NAMES,
    NUM_LABELS,
    SCENARIOS,
    TEXT_COLUMNS,
)

print("Labels:")
for i, label in enumerate(LABEL_NAMES, start=1):
    print(f"  {i}. {label}")

print("\nNumber of labels:", NUM_LABELS)

print("\nClinical text columns:")
for column in TEXT_COLUMNS:
    print(f"  - {column}")

print("\nFinal scenarios:")
for name, modalities in SCENARIOS.items():
    print(f"  {name}: {modalities}")

Labels:
  1. label_caries
  2. label_gingivitis
  3. label_malocclusion
  4. label_pulpitis
  5. label_tooth_loss
  6. label_tooth_structure_loss

Number of labels: 6

Clinical text columns:
  - chief_complaint
  - present_illness
  - past_medical_record
  - examination

Final scenarios:
  text_only: ('text',)
  image_only: ('photograph',)
  xray_only: ('radiograph',)
  image_text: ('photograph', 'text')
  image_xray: ('photograph', 'radiograph')
  text_xray: ('text', 'radiograph')
  full_multimodal: ('photograph', 'radiograph', 'text')


## 05.17 — Leakage Audit Status

A dataset-level leakage audit was performed before the final supervised experiments.

The patient-level split was checked to ensure that patients do not overlap between training, validation, and test partitions.

Image reuse and identifier overlap were also investigated.

The final patient-level split is considered safe for the supervised benchmark.

Text leakage was examined separately.

No explicit diagnostic target fields are provided to the model. However, the approved clinical fields naturally contain clinically informative findings.

Therefore, the correct interpretation is:

> The final benchmark avoids direct label-field leakage, but the clinical text modality may naturally encode information strongly associated with the diagnostic targets.

This is a property of the available clinical information rather than an implementation error, and it is one reason the text-only result is treated as a strong clinical-information reference baseline.


## 05.18 — Historical Experiment Policy

Several experiments were conducted during development using different dataset populations, training configurations, architectures, or checkpoint-selection rules.

Examples include:

* modality-specific populations rather than the common complete-case population;
* different learning rates;
* different hidden dimensions;
* different early-stopping patience values;
* unfrozen image encoders;
* earlier model architectures;
* pre-BatchNorm-freezing correction experiments.

These experiments were useful for debugging and model development but are **not part of the final seven-scenario benchmark**.

The final thesis comparison uses only the results generated under the corrected common protocol defined in Sections 05.6–05.10.

This distinction is important because a numerical improvement obtained under a different experimental protocol cannot be treated as a fair modality comparison.


## 05.19 — Final Seven-Scenario Benchmark Results

All results in this section were obtained using the **same final supervised protocol**, the same fixed patient-level split, and the same complete-case population.

The primary reported threshold is `0.5`.

### Primary test results

| Scenario                | Test visits | Macro F1 @ 0.5 | Micro F1 @ 0.5 |      AUROC |   Accuracy | Best epoch |
| ----------------------- | ----------: | -------------: | -------------: | ---------: | ---------: | ---------: |
| Full Multimodal         |         633 |     **0.8317** |         0.8879 |     0.9731 |     0.8246 |         15 |
| Text + Radiograph       |         633 |     **0.8221** |         0.8877 |     0.9687 |     0.8199 |         18 |
| Text-only               |         633 |     **0.8211** |         0.8975 | **0.9745** | **0.8357** |         10 |
| Photograph + Text       |         633 |         0.7928 |         0.8856 |     0.9721 |     0.8199 |         15 |
| Photograph + Radiograph |         633 |         0.4351 |         0.6875 |     0.8647 |     0.6271 |         46 |
| Photograph-only         |         633 |         0.3909 |         0.6774 |     0.8291 |     0.6066 |         50 |
| Radiograph-only         |         633 |         0.2794 |         0.5832 |     0.7770 |     0.5340 |         18 |

The primary comparison is based on **Macro F1 @ threshold 0.5** and **Macro AUROC**.

> Values shown as `—` are not required for the primary comparison table and are omitted here because the corresponding detailed results are not needed to establish the main modality comparison.


## 05.20 — Primary Result Interpretation

The final supervised benchmark shows a clear difference between the diagnostic information provided by the three modalities.

### 1. Clinical text is the strongest unimodal modality

The text-only model achieves:

* Macro F1: **0.8211**
* AUROC: **0.9745**

This is substantially stronger than either image-only or radiograph-only classification.

The result confirms that the approved clinical-history and examination fields contain a large amount of information predictive of the six diagnostic targets.

As discussed previously, this should be interpreted as a strong **clinical-information baseline**, not as a pure symptom-only diagnostic system.

### 2. Adding radiographs to text provides little additional benefit in the standard supervised setting

The Text + Radiograph model achieves:

* Macro F1: **0.8221**
* AUROC: **0.9687**

Its Macro F1 is almost identical to the text-only result.

This suggests that, under this supervised baseline protocol, radiographs do not provide a large additional gain beyond the information already available in the clinical text.

This observation is particularly relevant for the later investigation of radiograph availability and missing-radiograph robustness.

### 3. Full multimodal fusion gives the highest Macro F1

The Full Multimodal model achieves:

* Macro F1: **0.8317**
* AUROC: **0.9731**

It therefore obtains the highest Macro F1 among the seven final scenarios.

However, the improvement over text-only is relatively small:

$$
0.8317 - 0.8211 \approx 0.0106
$$

At the same time, the text-only model has slightly higher AUROC (`0.9745`) than the full multimodal model (`0.9731`).

Therefore, the results do **not** support the stronger claim that multimodal fusion universally outperforms the text modality.

The more accurate conclusion is:

> **Clinical text is the dominant information source in the supervised benchmark, while full multimodal fusion provides a modest Macro F1 improvement over text-only classification.**


## 05.21 — Photograph and Radiograph Results

The visual-only scenarios perform substantially below the text-based scenarios.

| Scenario                | Macro F1 @ 0.5 |  AUROC |
| ----------------------- | -------------: | -----: |
| Photograph-only         |         0.3909 | 0.8291 |
| Photograph + Radiograph |         0.4351 | 0.8647 |
| Radiograph-only         |         0.2794 | 0.7770 |

The photograph modality performs better than the radiograph modality when used alone.

Combining photographs and radiographs improves over either visual modality alone in terms of Macro F1 and AUROC, but the resulting performance remains substantially below the text-based scenarios.

The result suggests that, within this supervised setup, visual information alone is insufficient to match the diagnostic information available in the permitted clinical text.

This motivates the subsequent self-supervised experiments: rather than assuming that supervised visual classification is already optimal, the next stage investigates whether self-supervised multimodal representation learning can produce more useful visual and radiographic representations.


## 05.22 — Secondary Validation-Optimized Threshold Results

As a secondary analysis, thresholds were optimized independently for each label using the validation set only.

The resulting thresholds were then applied to the held-out test set.

| Scenario          | Test Macro F1 @ 0.5 | Test Macro F1 @ optimized thresholds | Test AUROC |
| ----------------- | ------------------: | -----------------------------------: | ---------: |
| Full Multimodal   |              0.8317 |                           **0.8443** |     0.9731 |
| Text + Radiograph |              0.8221 |                               0.8281 |     0.9687 |
| Text-only         |              0.8211 |                               0.8173 |     0.9745 |
| Radiograph-only   |              0.2794 |                               0.4236 |     0.7770 |

For the Full Multimodal model, the validation-derived thresholds were:

`[0.80, 0.10, 0.20, 0.30, 0.15, 0.50]`

in the order:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth loss
6. Tooth structure loss

The optimized-threshold results demonstrate that the fixed `0.5` threshold is not necessarily optimal for every label.

However, these results remain **secondary** because the primary benchmark intentionally uses the same fixed threshold of `0.5` for all scenarios.


## 05.23 — Best Checkpoints

The best model checkpoint for each scenario was selected exclusively according to **validation Macro F1 at the fixed threshold of 0.5**.

The final selected checkpoints are:

| Scenario                | Best epoch | Best validation Macro F1 |
| ----------------------- | ---------: | -----------------------: |
| Text-only               |         10 |                   0.8155 |
| Photograph-only         |         50 |                   0.4001 |
| Radiograph-only         |         18 |                   0.2664 |
| Photograph + Text       |         15 |               **0.8072** |
| Photograph + Radiograph |         46 |               **0.4008** |
| Text + Radiograph       |         18 |                   0.8475 |
| Full Multimodal         |         15 |                   0.8278 |

Checkpoint selection was performed independently for each scenario, but the **selection criterion and threshold were identical across all scenarios**.

Specifically, for each training epoch:

$$
\text{Validation Macro F1}
=
\text{MacroF1}(y,\hat{y}_{t=0.5})
$$

The checkpoint corresponding to the highest validation Macro F1 was retained.

The test set was not used during checkpoint selection.

After training, the selected checkpoint was loaded and the final validation and test evaluations were performed.

### Selected epochs

The selected epochs vary across scenarios because each model learns at a different rate and reaches its best validation performance at a different point.

The different selected epochs therefore do not represent different training protocols. All models were allowed up to 50 epochs and used the same early-stopping patience of 10 epochs.

### Validation results of the two visual multimodal scenarios

The Photograph + Text model reached a best validation Macro F1 of `0.8072` at epoch 15.

The Photograph + Radiograph model reached a best validation Macro F1 of `0.4008` at epoch 46.

These values are substantially different despite both scenarios being trained under the same optimization protocol, reflecting the different predictive strength of the available modality combinations.


## 05.24 — Baseline Benchmark Conclusion

The final supervised benchmark establishes three main empirical observations:

1. **Clinical text is the strongest individual modality.**
2. **Visual modalities alone perform substantially worse than text-based scenarios.**
3. **Full multimodal fusion provides a modest improvement in Macro F1 over text-only classification, but not in AUROC.**

Therefore, the supervised results provide a strong baseline but do not by themselves demonstrate a large benefit from radiographs.

This makes the subsequent representation-learning experiments important.

The next stage investigates whether self-supervised multimodal pretraining can improve the learned visual/radiographic representations and whether the resulting multimodal system can remain effective when one modality—particularly radiographs—is naturally unavailable.


## 05.25 — Per-Label Evaluation

In addition to aggregate performance, per-label metrics are recorded for the final supervised scenarios.

For each diagnostic label, the following metrics are available:

* F1 score
* Precision
* Recall
* AUROC

Per-label results are particularly useful because Macro F1 is calculated by giving equal weight to all six diagnostic targets, while the prevalence and difficulty of individual conditions can differ substantially.

The per-label analysis is therefore used to identify which diagnostic targets are responsible for the observed differences between modality configurations.


## 05.26 — Text-Only Per-Label Results

The final text-only model produces the following test-set results at threshold `0.5`:

| Label                |     F1 | Precision | Recall |  AUROC |
| -------------------- | -----: | --------: | -----: | -----: |
| Caries               | 0.8790 |    0.9079 | 0.8519 | 0.9670 |
| Gingivitis           | 0.9613 |    0.9775 | 0.9457 | 0.9993 |
| Malocclusion         | 0.9236 |    0.9145 | 0.9329 | 0.9624 |
| Pulpitis             | 0.7302 |    0.7931 | 0.6765 | 0.9907 |
| Tooth loss           | 0.7660 |    0.9474 | 0.6429 | 0.9477 |
| Tooth structure loss | 0.6667 |    0.6875 | 0.6471 | 0.9797 |

The text-only model performs particularly strongly for gingivitis and malocclusion.

The lowest F1 score is observed for tooth structure loss, followed by pulpitis and tooth loss.

Interestingly, several labels have very high AUROC despite lower F1 scores. This indicates that the model can rank positive and negative examples effectively while the fixed `0.5` threshold does not necessarily produce the optimal classification boundary for every label.


## 05.27 — Text + Radiograph Per-Label Results

The final Text + Radiograph model produces:

| Label                |     F1 | Precision | Recall |  AUROC |
| -------------------- | -----: | --------: | -----: | -----: |
| Caries               | 0.8571 |    0.9041 | 0.8148 | 0.9183 |
| Gingivitis           | 0.9622 |    0.9570 | 0.9674 | 0.9991 |
| Malocclusion         | 0.9097 |    0.9067 | 0.9128 | 0.9597 |
| Pulpitis             | 0.7941 |    0.7941 | 0.7941 | 0.9906 |
| Tooth loss           | 0.7755 |    0.9048 | 0.6786 | 0.9640 |
| Tooth structure loss | 0.6341 |    0.5417 | 0.7647 | 0.9804 |

The addition of radiographs to clinical text does not produce a consistent improvement across labels.

For example, pulpitis F1 improves from approximately `0.73` with text-only to approximately `0.79` with text + radiograph, while caries and malocclusion F1 decrease slightly.

This is consistent with the aggregate results showing that the radiograph provides only limited additional benefit when combined with clinical text in this supervised setting.


## 05.28 — Full Multimodal Per-Label Results

The final Full Multimodal model produces:

| Label                |     F1 | Precision | Recall |  AUROC |
| -------------------- | -----: | --------: | -----: | -----: |
| Caries               | 0.8519 |    0.8519 | 0.8519 | 0.9577 |
| Gingivitis           | 0.9617 |    0.9670 | 0.9565 | 0.9993 |
| Malocclusion         | 0.9058 |    0.8925 | 0.9195 | 0.9611 |
| Pulpitis             | 0.8286 |    0.8056 | 0.8529 | 0.9909 |
| Tooth loss           | 0.7200 |    0.8182 | 0.6429 | 0.9483 |
| Tooth structure loss | 0.7222 |    0.6842 | 0.7647 | 0.9811 |

The Full Multimodal model provides its strongest performance on gingivitis, malocclusion, and pulpitis.

Compared with the text-only model, the full model improves F1 for pulpitis and tooth structure loss, while performance decreases for caries, malocclusion, and tooth loss.

This reinforces the conclusion that multimodal fusion does not uniformly improve every diagnostic target.


## 05.29 — Radiograph-Only Per-Label Results

The Radiograph-only model produces:

| Label                |     F1 | Precision | Recall |  AUROC |
| -------------------- | -----: | --------: | -----: | -----: |
| Caries               | 0.0000 |    0.0000 | 0.0000 | 0.6107 |
| Gingivitis           | 0.4903 |    0.6032 | 0.4130 | 0.8012 |
| Malocclusion         | 0.7374 |    0.7175 | 0.7584 | 0.8363 |
| Pulpitis             | 0.4490 |    0.7333 | 0.3235 | 0.8933 |
| Tooth loss           | 0.0000 |    0.0000 | 0.0000 | 0.8009 |
| Tooth structure loss | 0.0000 |    0.0000 | 0.0000 | 0.7195 |

The radiograph-only model illustrates an important distinction between ranking performance and thresholded classification.

For several labels, the model has substantially better-than-random AUROC despite an F1 score of zero at threshold `0.5`.

This means that the radiographic representation contains predictive signal, but the default decision threshold is poorly calibrated for some labels.

This is one reason validation-optimized thresholds are retained as a secondary analysis.


## 05.30 — Per-Label Interpretation

The per-label results support several observations:

### Gingivitis

Gingivitis is consistently among the easiest targets across the strong text-based models, with AUROC close to `1.0` and F1 around `0.96`.

### Malocclusion

Malocclusion is also predicted relatively well by text-based models and remains one of the strongest labels for the visual/radiographic models.

### Pulpitis

Pulpitis shows a clearer benefit from multimodal information. Its F1 increases from approximately `0.73` in the text-only model to approximately `0.83` in the full multimodal model.

### Tooth loss

Tooth loss remains more difficult despite relatively high AUROC values. The gap between ranking performance and thresholded F1 suggests that classification threshold and calibration play an important role.

### Tooth structure loss

Tooth structure loss is one of the weaker labels for the text-only model but improves in the full multimodal model.

Overall, the effect of multimodal fusion is **label-dependent rather than uniformly beneficial**.


## 05.31 — Per-Label Analysis Conclusion

The per-label analysis demonstrates that aggregate Macro F1 alone does not fully describe the behavior of the multimodal system.

Different modalities contribute differently to different diagnostic targets.

In particular:

* clinical text provides strong predictive information across most labels;
* radiographs contain useful signal for several targets but perform poorly as a standalone classifier under the fixed `0.5` threshold;
* full multimodal fusion improves some labels, particularly pulpitis and tooth structure loss;
* other labels do not improve over the text-only baseline.

Therefore, the contribution of multimodal learning should be evaluated not only through overall performance but also through its effect on individual diagnostic targets.

The next stage moves from supervised baseline performance to **self-supervised multimodal representation learning**, where the objective is to investigate whether pretrained multimodal representations can improve the usefulness and robustness of visual and radiographic information.


## 05.32 — Final Supervised Baseline Summary

The final supervised benchmark establishes a controlled reference for the six-label diagnostic task on the COde dataset.

All seven scenarios were evaluated using:

* the same fixed patient-level split;
* the same complete-case population;
* the same six diagnostic labels;
* the same training configuration;
* the same checkpoint-selection criterion;
* the same primary decision threshold (`0.5`);
* the same evaluation protocol.

The resulting benchmark is therefore suitable for direct comparison across modality combinations.


## 05.33 — Final Benchmark Table

The final primary benchmark is summarized below.

| Rank by Macro F1 | Scenario                | Test Visits | Macro F1 @ 0.5 |      AUROC |
| ---------------: | ----------------------- | ----------: | -------------: | ---------: |
|                1 | Full Multimodal         |         633 |     **0.8317** |     0.9731 |
|                2 | Text + Radiograph       |         633 |     **0.8221** |     0.9687 |
|                3 | Text-only               |         633 |     **0.8211** | **0.9745** |
|                4 | Photograph + Text       |         633 |         0.7928 |     0.9721 |
|                5 | Photograph + Radiograph |         633 |         0.4351 |     0.8647 |
|                6 | Photograph-only         |         633 |         0.3909 |     0.8291 |
|                7 | Radiograph-only         |         633 |         0.2794 |     0.7770 |

### Main observations

The Full Multimodal model achieves the highest Macro F1 (`0.8317`).

The Text-only model achieves the highest AUROC (`0.9745`).

The difference between Full Multimodal and Text-only Macro F1 is approximately `0.0106`.

Text + Radiograph and Text-only have almost identical Macro F1 values.

The three scenarios relying only on visual information perform substantially worse than the text-based scenarios.


## 05.34 — Final Baseline Findings

The final supervised experiments support the following conclusions.

### Finding 1 — Clinical text is highly informative

The Text-only model achieves a Macro F1 of `0.8211` and AUROC of `0.9745`.

This demonstrates that the permitted clinical-history and examination fields contain substantial information associated with the six diagnostic targets.

### Finding 2 — Full multimodal fusion provides a modest Macro F1 gain

The Full Multimodal model achieves a Macro F1 of `0.8317`, compared with `0.8211` for Text-only.

The gain is approximately `1.06` percentage points in Macro F1.

However, the Full Multimodal model does not achieve the highest AUROC. Text-only reaches `0.9745`, compared with `0.9731` for Full Multimodal.

Therefore, the result should not be described as a universal multimodal superiority.

### Finding 3 — Radiographs add limited benefit when combined with text

Text + Radiograph achieves a Macro F1 of `0.8221`, which is only marginally higher than Text-only.

This indicates that the radiograph modality does not substantially improve aggregate supervised performance when clinical text is already available under this experimental setup.

### Finding 4 — Visual-only classification is substantially weaker

Photograph-only, Photograph + Radiograph, and Radiograph-only all perform below the text-based scenarios.

This demonstrates the difficulty of predicting the six diagnostic targets from visual information alone under the current supervised formulation.

### Finding 5 — Modality contribution is label-dependent

Per-label results show that multimodal fusion improves some diagnostic targets while producing little or no improvement for others.

Therefore, aggregate performance should be interpreted together with the per-label analysis.


## 05.35 — Methodological Validity of the Final Benchmark

The final benchmark satisfies the following methodological requirements:

* Patient-level separation between train, validation, and test sets.
* Identical evaluation population across all seven scenarios.
* No test-set checkpoint selection.
* No test-set threshold optimization.
* Fixed random seed (`42`).
* Identical optimizer and training hyperparameters across scenarios.
* Frozen pretrained image encoders.
* Explicit preservation of frozen ResNet BatchNorm layers in evaluation mode.
* Clinical-text input restricted to the approved fields.
* Exclusion of fields considered unsuitable because of direct or near-direct diagnostic information.
* Primary comparison based on a fixed threshold of `0.5`.
* Validation-optimized thresholds treated only as secondary analysis.

These controls make the final seven-scenario results appropriate as the canonical supervised baseline for the thesis.


## 05.36 — Final Status of Notebook 05

The supervised baseline experiments documented in this notebook are **complete**.

The canonical results are the results produced under the final corrected protocol described in this notebook.

Earlier exploratory results, alternative hyperparameter configurations, modality-specific populations, and pre-correction experiments are not considered final benchmark results.

The final supervised benchmark consists of exactly seven scenarios:

1. Text-only
2. Photograph-only
3. Radiograph-only
4. Photograph + Text
5. Photograph + Radiograph
6. Text + Radiograph
7. Full Multimodal

The primary reported metrics are:

* Test Macro F1 at threshold `0.5`
* Test Macro AUROC

Validation-optimized threshold results and per-label metrics are retained as secondary analyses.


In [16]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parents[1]
RESULT_ROOT = PROJECT_ROOT / "results" / "baseline" / "final"

SCENARIOS = [
    "text_only",
    "image_only",
    "xray_only",
    "image_text",
    "image_xray",
    "text_xray",
    "full_multimodal",
]

print("Final supervised baseline result files")
print("=" * 50)

for scenario in SCENARIOS:
    result_path = RESULT_ROOT / scenario / "final_results.json"

    if result_path.exists():
        with open(result_path, "r") as f:
            result = json.load(f)

        test_default = result["test_default"]

        print(
            f"{scenario:20s} | "
            f"Macro F1={test_default['macro_f1']:.4f} | "
            f"AUROC={test_default['auroc']:.4f}"
        )
    else:
        print(f"{scenario:20s} | MISSING RESULT FILE")

Final supervised baseline result files
text_only            | Macro F1=0.8211 | AUROC=0.9745
image_only           | Macro F1=0.3909 | AUROC=0.8291
xray_only            | Macro F1=0.2794 | AUROC=0.7770
image_text           | Macro F1=0.7928 | AUROC=0.9721
image_xray           | Macro F1=0.4351 | AUROC=0.8647
text_xray            | Macro F1=0.8221 | AUROC=0.9687
full_multimodal      | Macro F1=0.8317 | AUROC=0.9731


## 05.37 — Notebook Cleanup Notes

The following obsolete material should **not** remain as active final-baseline documentation:

* old modality-specific sample counts;
* old full multimodal architecture descriptions;
* old learning rates or hidden dimensions;
* old early-stopping settings;
* old results generated before the final BatchNorm correction;
* historical test results from different evaluation populations;
* exploratory hyperparameter comparisons;
* stale roadmap statements describing the supervised baseline as unfinished;
* obsolete claims based on the earlier `1792`-dimensional multimodal representation;
* unmarked debugging cells.

Any retained debugging or audit cells should be explicitly marked:

`# DELETE AFTER AUDIT`

The final notebook should present the corrected seven-scenario benchmark as the authoritative supervised baseline rather than as a chronological record of every development experiment.


## 05.38 — End of Supervised Baseline Notebook

**Status: Complete**

The final supervised benchmark has been established using a controlled patient-level evaluation protocol on the COde six-label diagnostic task.

The results and methodological decisions documented above constitute the final supervised baseline record.
